# Test du modèle avec un nouveau fichier CSV

In [1]:
import pandas as pd
import joblib
from datetime import datetime
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


## Chargement du modèle et des encoders

In [2]:
model = joblib.load("../models/lightgbm_model.pkl")
encoders = joblib.load("../models/encoders.pkl")
selected_features = joblib.load("../models/selected_features_LGB.pkl")
medians = joblib.load("../models/median_values.pkl")

c:\Users\pc\anaconda3\envs\flights-mlops\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Chargement du nouveau fichier csv

In [4]:
df_new = pd.read_csv("../data/test/flight_delays.csv")

print(df_new.head())

   FlightID    Airline  FlightNumber Origin Destination ScheduledDeparture  \
0         1     United          4558    ORD         MIA   2024-09-01 08:11   
1         2      Delta          8021    LAX         MIA   2024-09-01 10:25   
2         3  Southwest          7520    DFW         SFO   2024-09-01 16:53   
3         4      Delta          2046    ORD         BOS   2024-09-01 14:44   
4         5      Delta          6049    LAX         SEA   2024-09-01 01:51   

    ActualDeparture  ScheduledArrival     ActualArrival  DelayMinutes  \
0  2024-09-01 08:30  2024-09-01 12:11  2024-09-01 12:19             8   
1  2024-09-01 10:41  2024-09-01 13:25  2024-09-01 13:27             2   
2  2024-09-01 17:05  2024-09-01 17:53  2024-09-01 18:07            14   
3  2024-09-01 15:04  2024-09-01 18:44  2024-09-01 18:34           -10   
4  2024-09-01 02:08  2024-09-01 05:51  2024-09-01 06:15            24   

           DelayReason  Cancelled  Diverted AircraftType TailNumber  Distance  
0           

## Prétraitement des données 

In [5]:
df_new['flight_date'] = pd.to_datetime(df_new['ScheduledDeparture']).dt.date
df_new['crs_dep_hour'] = pd.to_datetime(df_new['ScheduledDeparture']).dt.hour
df_new['crs_dep_min'] = pd.to_datetime(df_new['ScheduledDeparture']).dt.minute
df_new['crs_arr_hour'] = pd.to_datetime(df_new['ScheduledArrival']).dt.hour
df_new['crs_arr_min'] = pd.to_datetime(df_new['ScheduledArrival']).dt.minute

# Jour, mois, jour de la semaine, weekend
df_new['fl_day'] = pd.to_datetime(df_new['flight_date']).dt.day
df_new['fl_month'] = pd.to_datetime(df_new['flight_date']).dt.month
df_new['fl_dayofweek'] = pd.to_datetime(df_new['flight_date']).dt.weekday
df_new['is_weekend'] = df_new['fl_dayofweek'].isin([5,6]).astype(int)

# Saison
def get_season(month):
    if month in [12,1,2]:
        return "winter"
    elif month in [3,4,5]:
        return "spring"
    elif month in [6,7,8]:
        return "summer"
    else:
        return "fall"
    
df_new['season'] = df_new['fl_month'].apply(get_season)




## Encodage des colonnes catégorielles

In [6]:
def safe_encode(encoder, val):
    val = str(val).strip()
    if val in encoder.classes_:
        return encoder.transform([val])[0]
    return -1

In [7]:
# Adapter noms de colonnes pour correspondre au modèle
df_new['op_carrier_fl_num'] = df_new['FlightNumber']  # map FlightNumber
df_new['origin_city_name'] = df_new['Origin'].apply(lambda x: safe_encode(encoders['origin_city_name'], x))
df_new['origin_state_nm'] = df_new['Origin'].apply(lambda x: safe_encode(encoders['origin_state_nm'], x))  # si vous avez l'état séparé
df_new['dest_city_name'] = df_new['Destination'].apply(lambda x: safe_encode(encoders['dest_city_name'], x))
df_new['dest_state_nm'] = df_new['Destination'].apply(lambda x: safe_encode(encoders['dest_state_nm'], x))

# Distance
df_new['distance'] = df_new['Distance']

# Temps prévu pour le vol (crs_elapsed_time)
df_new['crs_elapsed_time'] = ((pd.to_datetime(df_new['ScheduledArrival']) - pd.to_datetime(df_new['ScheduledDeparture'])).dt.total_seconds() / 60)

# Cible binaire
df_new['delay_label'] = (df_new['DelayMinutes'] > 0).astype(int)



## Création DataFrame pour les prédictions

In [8]:
# Remplir les colonnes manquantes avec les médianes
features = selected_features
X_new = pd.DataFrame()
for col in features:
    if col in df_new.columns:
        X_new[col] = df_new[col]
    else:
        X_new[col] = medians.get(col, 0)

In [9]:
X_new['season'] = X_new['fl_month'].apply(get_season).str.lower()
X_new['season'] = X_new['season'].apply(lambda x: safe_encode(encoders['season'], x))

# idem pour origin_city_name, dest_city_name, origin_state_nm, dest_state_nm
for col in ['origin_city_name','origin_state_nm','dest_city_name','dest_state_nm']:
    X_new[col] = X_new[col].apply(lambda x: safe_encode(encoders[col], x))

## Prédictions

In [10]:
y_pred = model.predict(X_new)
y_true = df_new['delay_label'].values

## Calcul du taux de réussite

In [11]:
# --- Metrics ---
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)  # % de prédictions retard correctes
recall = recall_score(y_true, y_pred)        # % de vrais retards détectés
f1 = f1_score(y_true, y_pred)                # moyenne harmonique precision/recal

print(f"Taux de réussite (accuracy) : {accuracy*100:.2f}%")
print(f"Précision : {precision*100:.2f}%")
print(f"Rappel : {recall*100:.2f}%")
print(f"F1-score : {f1*100:.2f}%")


Taux de réussite (accuracy) : 73.17%
Précision : 73.17%
Rappel : 100.00%
F1-score : 84.51%
